# 📊 Notebook 4 — RAG Evaluation Metrics

**DocuMind AI Portfolio Project**

1. Why evaluate RAG systems?
2. Creating test QA pairs
3. Retrieval metrics (Recall, Precision, MRR)
4. Generation metrics (ROUGE-L, keyword overlap)
5. Running full evaluation
6. Results visualisation
7. Recommendations for improvement

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'figure.facecolor':'#0d1117','axes.facecolor':'#161b22','text.color':'#e6edf3','axes.labelcolor':'#e6edf3','xtick.color':'#e6edf3','ytick.color':'#e6edf3'})
import numpy as np
import pandas as pd
from dotenv import load_dotenv
load_dotenv('../.env')
print('✅ Ready')

## 1. Why Evaluate RAG Systems?

Without evaluation, you can't know if your RAG system is actually answering questions correctly.

**Key failure modes to detect:**
- Retrieval failure: relevant document not found → wrong answer
- Faithfulness failure: LLM generates unsupported information (hallucination)
- Relevancy failure: retrieved chunks are returned but answer misses the point

**Key metrics:**
| Metric | What it measures | Target |
|--------|-----------------|--------|
| Recall@K | Did retrieval find the right document? | > 0.85 |
| MRR | How highly ranked is the correct document? | > 0.75 |
| ROUGE-L | N-gram overlap with ground truth | > 0.40 |
| Faithfulness | Is the answer grounded in context? | > 0.80 |

In [ ]:
from src.rag_pipeline import RAGPipeline
from src.evaluation import RAGEvaluator

pipeline = RAGPipeline()
pipeline.ingest_documents('data/raw/sample_docs')
evaluator = RAGEvaluator(pipeline, pipeline.config)
print('Evaluator ready.')

## 2. Test QA Pairs

In [ ]:
qa_pairs = evaluator.create_test_qa_pairs()
df = pd.DataFrame(qa_pairs)
print(f'Total test pairs: {len(qa_pairs)}')
print('\nSample pairs:')
print(df[['document', 'question']].head(10).to_string(index=False))

## 3. Retrieval Evaluation

In [ ]:
retrieval_metrics = evaluator.evaluate_retrieval(qa_pairs[:10])  # subset for speed
print('Retrieval Metrics:')
for k, v in retrieval_metrics.items():
    print(f'  {k}: {v}')

## 4. Generation Evaluation

In [ ]:
# Use a small subset for speed in the notebook
generation_metrics = evaluator.evaluate_generation(qa_pairs[:5])
print('Generation Metrics:')
for k, v in generation_metrics.items():
    print(f'  {k}: {v}')

## 5. Results Visualisation

In [ ]:
# Visualise metrics as a radar / bar chart
metric_names = ['Recall@K', 'Precision@K', 'MRR', 'ROUGE-L', 'Keyword Overlap']
scores = [
    retrieval_metrics.get('recall_at_k', 0),
    retrieval_metrics.get('precision_at_k', 0),
    retrieval_metrics.get('mrr', 0),
    generation_metrics.get('avg_rouge_l', 0),
    generation_metrics.get('avg_keyword_overlap', 0),
]
targets = [0.85, 0.70, 0.75, 0.40, 0.50]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(metric_names))
w = 0.35
rects1 = ax.bar(x - w/2, scores, w, label='Achieved', color='#7C3AED', alpha=0.85)
rects2 = ax.bar(x + w/2, targets, w, label='Target', color='#06B6D4', alpha=0.6)
ax.set_title('RAG Evaluation Metrics vs Targets')
ax.set_xticks(x); ax.set_xticklabels(metric_names, rotation=20, ha='right')
ax.set_ylim(0, 1.1); ax.set_ylabel('Score')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
fig.tight_layout(); plt.show()

## 6. Speed Benchmark

In [ ]:
bench = evaluator.benchmark_speed(n_queries=5)
print('Speed Benchmark (5 queries):')
for k, v in bench.items():
    print(f'  {k}: {v}')

## 7. Improvement Recommendations

Based on evaluation results:

**If Recall@K < 0.85:**
- ✅ Increase `top_k` from 5 to 8-10
- ✅ Use multi-query retrieval to broaden search
- ✅ Reduce chunk size to 512 for finer granularity
- ✅ Enable hybrid search (BM25 + semantic)

**If ROUGE-L < 0.40:**
- ✅ Switch from GPT-3.5 to GPT-4 for better generation
- ✅ Lower temperature (closer to 0.0) for factual responses
- ✅ Apply contextual compression to reduce context noise
- ✅ Improve prompt template with more specific instructions

**If latency > 3 seconds:**
- ✅ Use OpenAI ada-002 instead of HuggingFace for 5x faster embeddings
- ✅ Reduce `top_k` to 3-4
- ✅ Enable Redis caching for repeated queries
- ✅ Use `gpt-3.5-turbo` instead of `gpt-4` for 3x faster responses